# 🔄 00 — Datalake Sync (Local Scala Pipeline → IBM Cloud Object Storage)

**Pipeline:** Sincronización Multi-Capa · Medallion Architecture  
**Autor:** Federico Pfund  
**Fecha:** Abril 2026  

---

## Objetivo

Sincronizar el **datalake local** generado por el pipeline Scala (`batch-etl-scala`) hacia los
buckets de **IBM Cloud Object Storage**, respetando el formato nativo de cada capa:

| Capa | Formato Origen | Formato Destino COS | Estrategia |
|------|---------------|---------------------|------------|
| **RAW** | CSV | CSV (crudo, sin transformación) | Copia byte-level via `ibm_boto3` |
| **Bronze** | Parquet (Spark) | Parquet (validado) | Re-lectura Spark → coalesce → write |
| **Silver** | Parquet (Spark) | Parquet (validado) | Re-lectura Spark → coalesce → write |
| **Gold** | Delta Lake | Delta Lake (full) | Copia `_delta_log/` + data files preservando ACID |

### Arquitectura de Sincronización

```
LOCAL DATALAKE (Scala Pipeline)           IBM Cloud Object Storage
┌────────────────────────────────┐    ┌───────────────────────────────┐
│  datalake/                      │    │  IBM COS Buckets              │
│  ├─ raw/        (7 CSV)          │───▶│  datalake-raw-us-south       │
│  ├─ bronze/     (7 Parquet)      │───▶│  datalake-bronze-us-south    │
│  ├─ silver/     (8 Parquet)      │───▶│  datalake-silver-us-south    │
│  └─ gold/       (7 Delta)        │───▶│  datalake-gold-us-south      │
└────────────────────────────────┘    └───────────────────────────────┘
```

### Características Avanzadas

- **SyncManifest**: Registro JSON de cada operación con checksum MD5, timestamps y estado.
- **Detección de cambios**: Compara checksum local vs remoto; solo sincroniza archivos modificados.
- **Modo dual**: `ibm_boto3` para archivos crudos (RAW/Gold Delta logs), Spark para Parquet vali dado.
- **Reporte visual**: Dashboard de progreso con métricas por capa.
- **Idempotente**: Puede ejecutarse múltiples veces sin duplicar datos.

---
## 1. Inicialización y Configuración

In [11]:
# ============================================================================
# INICIALIZACION: SparkSession + IBM COS boto3 client + constantes
# ============================================================================
import sys, os, json, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict
from typing import Optional

warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import build_spark, cos_path, COS_CONFIG, BUCKETS


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (16, 5)
plt.rcParams['figure.dpi'] = 120

# -- Rutas locales --
DATALAKE_ROOT = Path(
    '/workspaces/data-engineer/transformation/spark-jobs/pipelines/batch-etl-scala/datalake'
)

LAYERS = {
    'raw':    {'local': DATALAKE_ROOT / 'raw',    'bucket': BUCKETS['raw'],    'format': 'csv'},
    'bronze': {'local': DATALAKE_ROOT / 'bronze', 'bucket': BUCKETS['bronze'], 'format': 'parquet'},
    'silver': {'local': DATALAKE_ROOT / 'silver', 'bucket': BUCKETS['silver'], 'format': 'parquet'},
    'gold':   {'local': DATALAKE_ROOT / 'gold',   'bucket': BUCKETS['gold'],   'format': 'delta'},
}

# -- Spark Session --
spark = build_spark('Datalake-Sync')
print(f'SparkSession activa | Version: {spark.version}')
print(f'   Datalake root: {DATALAKE_ROOT}')
print(f'   Capas: {list(LAYERS.keys())}')

SparkSession activa | Version: 3.5.4
   Datalake root: /workspaces/data-engineer/transformation/spark-jobs/pipelines/batch-etl-scala/datalake
   Capas: ['raw', 'bronze', 'silver', 'gold']


---
## 2. IBM COS Client + SyncManifest Engine

Motor de sincronización con:
- **`CosClient`**: wrapper sobre `ibm_boto3` para uploads con checksum.
- **`SyncManifest`**: dataclass que registra cada operación.
- **`SyncEngine`**: orquestador que decide la estrategia por capa.

In [21]:
# ============================================================================
# SYNC ENGINE: COS Client + Manifest + Strategy per layer
# ============================================================================
import ibm_boto3
from ibm_botocore.client import Config as BotoConfig


@dataclass
class SyncRecord:
    """Single file sync operation record."""
    layer: str
    table: str
    file_name: str
    local_path: str
    cos_key: str
    size_bytes: int
    md5: str
    strategy: str           # 'byte_copy' | 'spark_parquet' | 'delta_copy'
    status: str = 'pending' # 'pending' | 'synced' | 'skipped' | 'error'
    duration_ms: float = 0.0
    error_msg: Optional[str] = None
    synced_at: Optional[str] = None


@dataclass
class SyncManifest:
    """Full sync manifest with records per layer."""
    started_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    records: list = field(default_factory=list)
    
    def add(self, rec: SyncRecord):
        self.records.append(rec)
    
    def summary(self) -> dict:
        by_layer = {}
        for r in self.records:
            layer = r.layer
            if layer not in by_layer:
                by_layer[layer] = {'synced': 0, 'skipped': 0, 'error': 0, 'bytes': 0, 'ms': 0}
            by_layer[layer][r.status] = by_layer[layer].get(r.status, 0) + 1
            by_layer[layer]['bytes'] += r.size_bytes
            by_layer[layer]['ms'] += r.duration_ms
        return by_layer
    
    def to_json(self) -> str:
        return json.dumps({
            'started_at': self.started_at,
            'completed_at': datetime.now(timezone.utc).isoformat(),
            'total_records': len(self.records),
            'summary': self.summary(),
            'records': [asdict(r) for r in self.records],
        }, indent=2, default=str)


class CosClient:
    """IBM COS S3-compatible client with checksum-based change detection."""

    def __init__(self):
        self._client = ibm_boto3.client(
            's3',
            aws_access_key_id=COS_CONFIG['access_key'],
            aws_secret_access_key=COS_CONFIG['secret_key'],
            endpoint_url=f"https://{COS_CONFIG['endpoint']}",
            config=BotoConfig(signature_version='s3v4'),
        )
    
    def object_exists(self, bucket: str, key: str) -> bool:
        try:
            self._client.head_object(Bucket=bucket, Key=key)
            return True
        except Exception:
            return False
    
    def get_remote_etag(self, bucket: str, key: str) -> Optional[str]:
        try:
            resp = self._client.head_object(Bucket=bucket, Key=key)
            return resp.get('ETag', '').strip('"')
        except Exception:
            return None
    
    def upload_file(self, local_path: str, bucket: str, key: str) -> dict:
        self._client.upload_file(local_path, bucket, key)
        resp = self._client.head_object(Bucket=bucket, Key=key)
        return {'etag': resp.get('ETag', '').strip('"'), 'size': resp.get('ContentLength', 0)}
    
    def list_keys(self, bucket: str, prefix: str = '') -> list:
        keys = []
        paginator = self._client.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get('Contents', []):
                keys.append(obj['Key'])
        return keys


def md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


cos = CosClient()
manifest = SyncManifest()
print('COS Client inicializado')
print(f'SyncManifest creado @ {manifest.started_at}')

COS Client inicializado
SyncManifest creado @ 2026-04-12T22:13:45.371167+00:00


---
## 3. Inventario Local del Datalake

Escaneo completo del datalake local para generar el inventario de archivos por capa,
con tamaño, formato y checksum MD5.

In [22]:
# ============================================================================
# INVENTARIO: Scan del datalake local con metricas
# ============================================================================
def scan_local_datalake(layers: dict) -> pd.DataFrame:
    """Scan all layers and build an inventory DataFrame."""
    rows = []
    for layer_name, cfg in layers.items():
        local_dir = cfg['local']
        if not local_dir.exists():
            print(f'  WARN: {local_dir} no existe, saltando')
            continue
            
        if layer_name == 'raw':
            # RAW: archivos CSV sueltos
            for f in sorted(local_dir.glob('*.csv')):
                rows.append({
                    'layer': layer_name,
                    'table': f.stem,
                    'file': f.name,
                    'path': str(f),
                    'size_bytes': f.stat().st_size,
                    'format': 'csv',
                })
        else:
            # BRONZE/SILVER/GOLD: subdirectorios por tabla
            for table_dir in sorted(local_dir.iterdir()):
                if not table_dir.is_dir():
                    continue
                for f in sorted(table_dir.rglob('*')):
                    if f.is_file() and not f.name.startswith('.'):
                        rel = f.relative_to(table_dir)
                        fmt = cfg['format']
                        rows.append({
                            'layer': layer_name,
                            'table': table_dir.name,
                            'file': str(rel),
                            'path': str(f),
                            'size_bytes': f.stat().st_size,
                            'format': fmt,
                        })
    return pd.DataFrame(rows)


inventory = scan_local_datalake(LAYERS)

# -- Resumen --
print('=' * 60)
print('  INVENTARIO DEL DATALAKE LOCAL')
print('=' * 60)

inv_summary = (
    inventory
    .groupby('layer')
    .agg(
        tablas=('table', 'nunique'),
        archivos=('file', 'count'),
        bytes_total=('size_bytes', 'sum'),
    )
    .reset_index()
)
inv_summary['size_mb'] = (inv_summary['bytes_total'] / 1024 / 1024).round(2)

for _, row in inv_summary.iterrows():
    print(f'  {row["layer"]:>8s}: {row["tablas"]:>2} tablas | '
          f'{row["archivos"]:>3} archivos | {row["size_mb"]:>8.2f} MB')

total_files = len(inventory)
total_mb = inventory['size_bytes'].sum() / 1024 / 1024
print(f'\n  TOTAL: {total_files} archivos | {total_mb:.2f} MB')
print('=' * 60)

display(inv_summary)

  INVENTARIO DEL DATALAKE LOCAL
    bronze:  7 tablas |  14 archivos |     0.77 MB
      gold:  7 tablas |  14 archivos |     1.22 MB
       raw:  7 tablas |   7 archivos |     6.63 MB
    silver:  8 tablas |  16 archivos |     1.15 MB

  TOTAL: 51 archivos | 9.78 MB


,layer,tablas,archivos,bytes_total,size_mb
0,bronze,7,14,806434,0.77
1,gold,7,14,1283507,1.22
2,raw,7,7,6956883,6.63
3,silver,8,16,1203899,1.15


---
## 4. Sync RAW Layer — Copia byte-level de CSV

Los archivos CSV crudos se copian **sin transformación** al bucket `datalake-raw-us-south`
usando `ibm_boto3`. Se aplica detección de cambios por checksum MD5: si el archivo remoto
ya tiene el mismo hash, se salta.

In [23]:
# ============================================================================
# SYNC RAW: Copia byte-level de CSVs crudos a COS
# ============================================================================
import time

print('\n' + '=' * 60)
print('  SYNC RAW LAYER \u2192 IBM COS (byte-copy + checksum)')
print('=' * 60)

raw_files = inventory[inventory['layer'] == 'raw']
bucket_raw = LAYERS['raw']['bucket']

for _, row in raw_files.iterrows():
    t0 = time.time()
    local_md5 = md5_file(row['path'])
    cos_key = f"{row['file']}"
    
    rec = SyncRecord(
        layer='raw', table=row['table'], file_name=row['file'],
        local_path=row['path'], cos_key=cos_key,
        size_bytes=row['size_bytes'], md5=local_md5, strategy='byte_copy'
    )
    
    # Deteccion de cambios via ETag
    remote_etag = cos.get_remote_etag(bucket_raw, cos_key)
    if remote_etag and remote_etag == local_md5:
        rec.status = 'skipped'
        rec.duration_ms = (time.time() - t0) * 1000
        print(f'  \u23ed  SKIP {row["file"]:>25s} | MD5 match (sin cambios)')
    else:
        try:
            result = cos.upload_file(row['path'], bucket_raw, cos_key)
            rec.status = 'synced'
            rec.synced_at = datetime.now(timezone.utc).isoformat()
            rec.duration_ms = (time.time() - t0) * 1000
            print(f'  \u2705  SYNC {row["file"]:>25s} | '
                  f'{row["size_bytes"]/1024:.1f} KB | {rec.duration_ms:.0f}ms')
        except Exception as e:
            rec.status = 'error'
            rec.error_msg = str(e)
            rec.duration_ms = (time.time() - t0) * 1000
            print(f'  \u274c  ERR  {row["file"]:>25s} | {e}')
    
    manifest.add(rec)

raw_synced = sum(1 for r in manifest.records if r.layer == 'raw' and r.status == 'synced')
raw_skipped = sum(1 for r in manifest.records if r.layer == 'raw' and r.status == 'skipped')
print(f'\n  RAW: {raw_synced} sincronizados | {raw_skipped} sin cambios | '
      f'{len(raw_files)} total')


  SYNC RAW LAYER → IBM COS (byte-copy + checksum)
  ⏭  SKIP             Categoria.csv | MD5 match (sin cambios)
  ⏭  SKIP              FactMine.csv | MD5 match (sin cambios)
  ⏭  SKIP                  Mine.csv | MD5 match (sin cambios)
  ⏭  SKIP              Producto.csv | MD5 match (sin cambios)
  ⏭  SKIP          Subcategoria.csv | MD5 match (sin cambios)
  ⏭  SKIP            Sucursales.csv | MD5 match (sin cambios)
  ⏭  SKIP        VentasInternet.csv | MD5 match (sin cambios)

  RAW: 0 sincronizados | 7 sin cambios | 7 total


---
## 5. Sync BRONZE Layer — Spark Parquet Validado

Las tablas Bronze se **re-leen con Spark** desde el filesystem local, se valida que estén
en Parquet válido (schema + conteo), y se escriben al bucket Bronze con `coalesce(1)` para
optimizar el tamaño de archivo en COS.

In [24]:
# ============================================================================
# SYNC BRONZE: Spark read local Parquet -> validate -> write COS Parquet
# ============================================================================
print('\n' + '=' * 60)
print('  SYNC BRONZE LAYER \u2192 IBM COS (Spark Parquet validado)')
print('=' * 60)

bronze_tables = sorted([
    d.name for d in LAYERS['bronze']['local'].iterdir() if d.is_dir()
])
bronze_stats = []

for table in bronze_tables:
    t0 = time.time()
    local_path = str(LAYERS['bronze']['local'] / table)
    cos_dest = cos_path('bronze', table)
    
    try:
        df = spark.read.parquet(local_path)
        row_count = df.count()
        col_count = len(df.columns)
        
        # Validacion: debe tener filas y columnas
        if row_count == 0:
            raise ValueError(f'Tabla {table} esta vacia (0 filas)')
        
        # Escribir a COS como Parquet optimizado
        df.coalesce(1).write.mode('overwrite').parquet(cos_dest)
        
        elapsed = (time.time() - t0) * 1000
        bronze_stats.append({
            'table': table, 'rows': row_count, 'cols': col_count,
            'ms': elapsed, 'status': 'synced'
        })
        
        # Registrar cada archivo del directorio local en el manifest
        for f in Path(local_path).rglob('*.parquet'):
            manifest.add(SyncRecord(
                layer='bronze', table=table, file_name=f.name,
                local_path=str(f), cos_key=f'{table}/{f.name}',
                size_bytes=f.stat().st_size, md5='spark-managed',
                strategy='spark_parquet', status='synced',
                duration_ms=elapsed,
                synced_at=datetime.now(timezone.utc).isoformat()
            ))
        
        print(f'  \u2705  {table:>20s} | {row_count:>6,} filas x {col_count:>2} cols | '
              f'{elapsed:>6.0f}ms')
    except Exception as e:
        elapsed = (time.time() - t0) * 1000
        bronze_stats.append({
            'table': table, 'rows': 0, 'cols': 0,
            'ms': elapsed, 'status': 'error'
        })
        manifest.add(SyncRecord(
            layer='bronze', table=table, file_name='',
            local_path=local_path, cos_key=f'{table}/',
            size_bytes=0, md5='', strategy='spark_parquet',
            status='error', error_msg=str(e), duration_ms=elapsed
        ))
        print(f'  \u274c  {table:>20s} | ERROR: {e}')

synced = sum(1 for s in bronze_stats if s['status'] == 'synced')
total_rows = sum(s['rows'] for s in bronze_stats)
print(f'\n  BRONZE: {synced}/{len(bronze_tables)} tablas | '
      f'{total_rows:,} filas totales')


  SYNC BRONZE LAYER → IBM COS (Spark Parquet validado)


  ✅             categoria |      4 filas x  4 cols |   3480ms


  ✅              factmine |     49 filas x  8 cols |   3322ms


  ✅                  mine | 15,205 filas x 13 cols |   3346ms


  ✅              producto |    319 filas x  6 cols |   3237ms


  ✅          subcategoria |     37 filas x  5 cols |   3121ms


  ✅            sucursales |     11 filas x  7 cols |   3180ms


  ✅        ventasinternet | 47,263 filas x 15 cols |   3718ms

  BRONZE: 7/7 tablas | 62,888 filas totales


---
## 6. Sync SILVER Layer — Spark Parquet Validado

Misma estrategia que Bronze: validación Spark + write optimizado a COS.

In [25]:
# ============================================================================
# SYNC SILVER: Spark read local Parquet -> validate -> write COS Parquet
# ============================================================================
print('\n' + '=' * 60)
print('  SYNC SILVER LAYER \u2192 IBM COS (Spark Parquet validado)')
print('=' * 60)

silver_tables = sorted([
    d.name for d in LAYERS['silver']['local'].iterdir() if d.is_dir()
])
silver_stats = []

for table in silver_tables:
    t0 = time.time()
    local_path = str(LAYERS['silver']['local'] / table)
    cos_dest = cos_path('silver', table)
    
    try:
        df = spark.read.parquet(local_path)
        row_count = df.count()
        col_count = len(df.columns)
        
        if row_count == 0:
            raise ValueError(f'Tabla {table} esta vacia (0 filas)')
        
        df.coalesce(1).write.mode('overwrite').parquet(cos_dest)
        
        elapsed = (time.time() - t0) * 1000
        silver_stats.append({
            'table': table, 'rows': row_count, 'cols': col_count,
            'ms': elapsed, 'status': 'synced'
        })
        
        for f in Path(local_path).rglob('*.parquet'):
            manifest.add(SyncRecord(
                layer='silver', table=table, file_name=f.name,
                local_path=str(f), cos_key=f'{table}/{f.name}',
                size_bytes=f.stat().st_size, md5='spark-managed',
                strategy='spark_parquet', status='synced',
                duration_ms=elapsed,
                synced_at=datetime.now(timezone.utc).isoformat()
            ))
        
        print(f'  \u2705  {table:>30s} | {row_count:>6,} filas x {col_count:>2} cols | '
              f'{elapsed:>6.0f}ms')
    except Exception as e:
        elapsed = (time.time() - t0) * 1000
        silver_stats.append({
            'table': table, 'rows': 0, 'cols': 0,
            'ms': elapsed, 'status': 'error'
        })
        manifest.add(SyncRecord(
            layer='silver', table=table, file_name='',
            local_path=local_path, cos_key=f'{table}/',
            size_bytes=0, md5='', strategy='spark_parquet',
            status='error', error_msg=str(e), duration_ms=elapsed
        ))
        print(f'  \u274c  {table:>30s} | ERROR: {e}')

synced = sum(1 for s in silver_stats if s['status'] == 'synced')
total_rows = sum(s['rows'] for s in silver_stats)
print(f'\n  SILVER: {synced}/{len(silver_tables)} tablas | '
      f'{total_rows:,} filas totales')


  SYNC SILVER LAYER → IBM COS (Spark Parquet validado)


  ✅              catalogo_productos |    319 filas x  7 cols |   3677ms


  ✅               eficiencia_minera |      7 filas x 10 cols |   3505ms


  ✅             produccion_operador |  9,132 filas x 11 cols |   3483ms


  ✅             produccion_por_pais |      6 filas x 10 cols |   3280ms


  ✅           rentabilidad_producto |    147 filas x 12 cols |   4273ms


  ✅        resumen_ventas_mensuales |     65 filas x 10 cols |   3332ms


  ✅           segmentacion_clientes | 17,555 filas x  9 cols |   3704ms


  ✅             ventas_enriquecidas | 47,263 filas x 26 cols |   3870ms

  SILVER: 8/8 tablas | 74,494 filas totales
